# DataPulse AI — Vertex AI Gemini with Google Cloud trial

This notebook uses **Vertex AI** and your Google Cloud project, **not** the AI Studio prepaid API key. It does not require a `GOOGLE_API_KEY`. Cloud calls may consume eligible trial credits. Do not upgrade billing automatically.


## 1 — Fetch the project and install SDK


In [ ]:
import os, subprocess
from pathlib import Path
repo = Path('/content/aegis-real-time-supply-chain-ai')
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git',str(repo)],check=True)
os.chdir(repo / 'datapulse-ai')
print('Project directory:',os.getcwd())
%pip -q install -r requirements.txt google-genai


## 2 — Authenticate and enter YOUR project ID

First confirm your project still has trial credits and [enable Vertex AI API](https://console.cloud.google.com/apis/library/aiplatform.googleapis.com). Your signed-in Google account needs access to call Vertex AI (typically Vertex AI User). No service-account key file is needed.


In [ ]:
from google.colab import auth
from google import genai

auth.authenticate_user()
PROJECT_ID = 'REPLACE_WITH_YOUR_GCP_PROJECT_ID'  # Project ID, not display name
LOCATION = 'global'  # Use global endpoint for broad model availability
assert PROJECT_ID != 'REPLACE_WITH_YOUR_GCP_PROJECT_ID', 'Set your actual project ID first'
vertex_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
print('Vertex AI client configured for project:', PROJECT_ID)


## 3 — Single low-volume Vertex AI test

A model can be unavailable in a project or region. If this fails, read the status and message before changing models. Cloud charges may apply. No AI Studio prepaid API key is used.


In [ ]:
MODEL_NAME = 'gemini-3.8-flash'
try:
    connection_test = vertex_client.models.generate_content(
        model=MODEL_NAME,
        contents='Reply with only: DataPulse Vertex AI connected.'
    )
    print('VERTEX AI RESPONSE:', connection_test.text)
except Exception as exc:
    print('VERTEX AI ERROR:', type(exc).__name__)
    print(str(exc)[:1800])
    raise


## 4 — Rebuild the verified DataPulse incident report

The ML results and root-cause report are constructed locally. The following call makes **no** Vertex AI request.


In [ ]:
import json
from datapulse.replay import decision_replay
from datapulse.investigator import investigate
summary, comparisons = decision_replay()
report = investigate(summary)
assert isinstance(report,dict) and 'evidence' in report and 'incident_type' in report
print(json.dumps(report,indent=2))


## 5 — Ask Vertex AI Gemini to explain the real report

This makes **one** additional cloud model request. Evidence-based explanation only; no autonomous database access or remediation.


In [ ]:
prompt = ('You are a read-only DataPulse AI incident assistant. Explain the verified JSON incident evidence, '
          'why basic validation missed the semantic shift, model decision impact, next checks, and human-approved actions. '
          'Do not invent figures, claim proven causal business harm, or treat synthetic outcomes as real customers.\n\n'
          + json.dumps(report, sort_keys=True))
response = vertex_client.models.generate_content(model=MODEL_NAME, contents=prompt)
print('DATAPULSE AI / VERTEX AI EXPLANATION\n')
print(response.text)


## Troubleshooting

- `403`: Vertex AI API disabled, IAM access, or trial restrictions
- `404`: model name/region or project model availability; try another supported model using [official model documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models)
- `429`: quota or rate limit
- Billing messages: review the **Google Cloud project** billing, not AI Studio prepaid balance
- `GOOGLE_API_KEY` must **not** be needed here

The developer Gemini API and Vertex AI have different authentication and billing paths. Never paste access tokens or API keys into chat or GitHub.
